In [ ]:
import tensorflow as tf
import numpy as np
from keras.models import *
from keras.layers import *
from keras.optimizers import *
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras import backend as keras
from tensorflow.keras.models import load_model as load_initial_model

from utils.metrics import dice_coef, jacard


def UNet(pretrained_weights = None,input_size = (608,576,1)):
    inputs = Input(input_size)
    conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(inputs)
    conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool1)
    conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool2)
    conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
    conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool3)
    conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv4)
    drop4 = Dropout(0.5)(conv4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(drop4)

    conv5 = Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool4)
    conv5 = Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv5)
    drop5 = Dropout(0.5)(conv5)

    up6 = Conv2D(512, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(drop5))
    merge6 = concatenate([drop4,up6], axis = 3)
    conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge6)
    conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv6)

    up7 = Conv2D(256, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv6))
    merge7 = concatenate([conv3,up7], axis = 3)
    conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge7)
    conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv7)

    up8 = Conv2D(128, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv7))
    merge8 = concatenate([conv2,up8], axis = 3)
    conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge8)
    conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv8)

    up9 = Conv2D(64, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv8))
    merge9 = concatenate([conv1,up9], axis = 3)
    conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge9)
    conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)
    conv9 = Conv2D(2, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)
    conv10 = Conv2D(1, 1, activation = 'sigmoid')(conv9)

    model = Model(inputs,conv10)

    model.compile(optimizer = Adam(lr = 1e-4), loss = 'binary_crossentropy', metrics = ['accuracy',dice_coef,jacard, tf.keras.metrics.AUC(), tf.keras.metrics.MeanIoU(num_classes=2),
                                                                                        tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    #print(model.summary()

    if(pretrained_weights):
        model.load_weights(pretrained_weights)

    return model

In [ ]:
from keras.layers import Conv2D, MaxPooling2D, UpSampling2D, BatchNormalization, Reshape, Permute, Activation, Input, \
    add, multiply
from keras.layers import concatenate, core, Dropout
from keras.models import Model
from keras.layers.merge import concatenate
from keras.optimizers import Adam
from keras.optimizers import SGD
from keras.layers.core import Lambda
import keras.backend as K

from utils.metrics import dice_coef, jacard


def up_and_concate(down_layer, layer):

    up = UpSampling2D(size=(2, 2))(down_layer)
    my_concat = Lambda(lambda x: K.concatenate([x[0], x[1]], axis=3))
    concate = my_concat([up, layer])

    return concate


def attention_up_and_concate(down_layer, layer):
    in_channel = down_layer.get_shape().as_list()[3]

    up = UpSampling2D(size=(2, 2))(down_layer)
    layer = attention_block_2d(x=layer, g=up, inter_channel=in_channel // 4)
    my_concat = Lambda(lambda x: K.concatenate([x[0], x[1]], axis=1))
    my_concat = Lambda(lambda x: K.concatenate([x[0], x[1]], axis=3))

    concate = my_concat([up, layer])
    return concate


def attention_block_2d(x, g, inter_channel):
    theta_x = Conv2D(inter_channel, [1, 1], strides=[1, 1])(x)

    phi_g = Conv2D(inter_channel, [1, 1], strides=[1, 1])(g)

    f = Activation('relu')(add([theta_x, phi_g]))
    psi_f = Conv2D(1, [1, 1], strides=[1, 1])(f)

    rate = Activation('sigmoid')(psi_f)

    att_x = multiply([x, rate])

    return att_x

def AttentionUNet(input_size = (608,576,1)):
    inputs = Input(input_size)
    x = inputs
    depth = 4
    features = 64
    skips = []
    for i in range(depth):
        x = Conv2D(features, (3, 3), activation='relu', padding='same')(x)
        x = Dropout(0.2)(x)
        x = Conv2D(features, (3, 3), activation='relu', padding='same')(x)
        skips.append(x)
        x = MaxPooling2D((2, 2))(x)
        features = features * 2

    x = Conv2D(features, (3, 3), activation='relu', padding='same')(x)
    x = Dropout(0.2)(x)
    x = Conv2D(features, (3, 3), activation='relu', padding='same')(x)

    for i in reversed(range(depth)):
        features = features // 2
        x = attention_up_and_concate(x, skips[i])
        x = Conv2D(features, (3, 3), activation='relu', padding='same')(x)
        x = Dropout(0.2)(x)
        x = Conv2D(features, (3, 3), activation='relu', padding='same')(x)

    conv6 = Conv2D(1, (1, 1), padding='same')(x)
    conv7 = core.Activation('sigmoid')(conv6)
    model = Model(inputs=inputs, outputs=conv7)

    model.compile(optimizer = Adam(lr = 1e-4), loss = 'binary_crossentropy', metrics = ['accuracy',dice_coef,jacard, tf.keras.metrics.AUC(), tf.keras.metrics.MeanIoU(num_classes=2),
                                                                                        tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

In [ ]:
import numpy as np
import keras
import tensorflow as tf
from keras.models import *
from keras.layers import *

from keras.optimizers import Adam, SDG
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras import backend as K
from tensorflow.keras.models import load_model as load_initial_model
from keras.losses import binary_crossentropy


from utils.metrics import dice_coef, jacard

def fire_module(x, fire_id, squeeze=16, expand=64):
    f_name = "fire{0}/{1}"
    channel_axis = 1 if K.image_data_format() == 'channels_first' else -1

    x = Conv2D(squeeze, (1, 1), activation='relu', padding='same', name=f_name.format(fire_id, "squeeze1x1"))(x)
    x = BatchNormalization(axis=channel_axis)(x)

    left = Conv2D(expand, (1, 1), activation='relu', padding='same', name=f_name.format(fire_id, "expand1x1"))(x)
    right = Conv2D(expand, (3, 3), activation='relu', padding='same', name=f_name.format(fire_id, "expand3x3"))(x)
    x = concatenate([left, right], axis=channel_axis, name=f_name.format(fire_id, "concat"))
    return x


def SqueezeUNet(inputs, num_classes=None, deconv_ksize=3, dropout=0.5, activation='sigmoid'):

    channel_axis = 1 if K.image_data_format() == 'channels_first' else -1
    if num_classes is None:
        num_classes = K.int_shape(inputs)[channel_axis]

    x01 = Conv2D(64, (3, 3), strides=(2, 2), padding='same', activation='relu', name='conv1')(inputs)
    x02 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), name='pool1', padding='same')(x01)

    x03 = fire_module(x02, fire_id=2, squeeze=16, expand=64)
    x04 = fire_module(x03, fire_id=3, squeeze=16, expand=64)
    x05 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), name='pool3', padding="same")(x04)

    x06 = fire_module(x05, fire_id=4, squeeze=32, expand=128)
    x07 = fire_module(x06, fire_id=5, squeeze=32, expand=128)
    x08 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), name='pool5', padding="same")(x07)

    x09 = fire_module(x08, fire_id=6, squeeze=48, expand=192)
    x10 = fire_module(x09, fire_id=7, squeeze=48, expand=192)
    x11 = fire_module(x10, fire_id=8, squeeze=64, expand=256)
    x12 = fire_module(x11, fire_id=9, squeeze=64, expand=256)

    if dropout != 0.0:
        x12 = Dropout(dropout)(x12)

    up1 = concatenate([
        Conv2DTranspose(192, deconv_ksize, strides=(1, 1), padding='same')(x12),
        x10,
    ], axis=channel_axis)
    up1 = fire_module(up1, fire_id=10, squeeze=48, expand=192)

    up2 = concatenate([
        Conv2DTranspose(128, deconv_ksize, strides=(1, 1), padding='same')(up1),
        x08,
    ], axis=channel_axis)
    up2 = fire_module(up2, fire_id=11, squeeze=32, expand=128)

    up3 = concatenate([
        Conv2DTranspose(64, deconv_ksize, strides=(2, 2), padding='same')(up2),
        x05,
    ], axis=channel_axis)
    up3 = fire_module(up3, fire_id=12, squeeze=16, expand=64)

    up4 = concatenate([
        Conv2DTranspose(32, deconv_ksize, strides=(2, 2), padding='same')(up3),
        x02,
    ], axis=channel_axis)
    up4 = fire_module(up4, fire_id=13, squeeze=16, expand=32)
    up4 = UpSampling2D(size=(2, 2))(up4)

    x = concatenate([up4, x01], axis=channel_axis)
    x = Conv2D(64, (3, 3), strides=(1, 1), padding='same', activation='relu')(x)
    x = UpSampling2D(size=(2, 2))(x)
    x = Conv2D(num_classes, (1, 1), activation=activation)(x)
    model = Model(inputs, x)
    #model.summary()
    model.compile(optimizer=Adam(lr = 1e-4), loss=binary_crossentropy, \
                  metrics = ['accuracy',dice_coef,jacard,tf.keras.metrics.MeanIoU(num_classes=2),\
                             tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

    return model

In [ ]:
import os, argparse, shutil, cv2, pickle, time, logging, gc, json

logging.disable(logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import skimage.io as io
import skimage.transform as trans
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

from keras.preprocessing.image import ImageDataGenerator
from keras.models import *
from keras.layers import *
from keras.optimizers import *
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras import backend as keras
from tensorflow.keras.models import load_model as load_initial_model


from unet.attention_unet import AttentionUNet
from unet.vanilla_unet import UNet
from utils.metrics import dice, mean_dice
from utils.preprocess import adjustData, pad, crop, threshold
from utils.file import get_dirs, set_order, saveResult_drive, saveResult
from data.generators import trainGenerator, testGenerator, testGenerator2

dirs = get_dirs()
LOG_PATH     = dirs["files"][0]["LOG_PATH"]
RESULT_PATH  = dirs["files"][0]["RESULT_PATH"]
MODEL_PATH   = dirs["files"][0]["MODEL_PATH"]

TRAIN_PATH   = dirs["files"][0]["TRAIN_PATH"]
TEST_PATH    = dirs["files"][0]["TEST_PATH"]
VAL_PATH     = dirs["files"][0]["VAL_PATH"]

TMP_TRAIN    = dirs["files"][0]["TMP_TRAIN"]
TMP_TEST     = dirs["files"][0]["TMP_TEST"]
TMP_VAL      = dirs["files"][0]["TMP_VAL"]
TMP_RESULT   = dirs["files"][0]["TMP_RESULT"]



TRAIN_PATH_IMG    = dirs["files"][0]["TRAIN_PATH"] + "/images"
TRAIN_PATH_MASK   = dirs["files"][0]["TRAIN_PATH"] + "/labels"
KFOLD_TEMP_TRAIN  = dirs["files"][0]["KFOLD_TEMP_TRAIN"]
KFOLD_TEMP_TEST   = dirs["files"][0]["KFOLD_TEMP_TEST"]

LOG_PATH_K        = dirs["files"][0]["LOG_PATH_KFOLD"]
CKPTS_PATH        = dirs["files"][0]["CKPTS_PATH_KFOLD"]
RESULTS_PATH      = dirs["files"][0]["RESULTS_PATH_KFOLD"]


def train_once(save_name, num_train, num_test, initial_model_path,\
               train_batch = 3, test_batch = 3, epoch = 5, already_padded = False,\
               model_name = "vanilla"):

    shutil.rmtree(TMP_TRAIN, ignore_errors=False, onerror=None)
    os.mkdir(TMP_TRAIN)
    os.mkdir(TMP_TRAIN + "/images")
    os.mkdir(TMP_TRAIN + "/labels")

    shutil.rmtree(TMP_TEST, ignore_errors=False, onerror=None)
    os.mkdir(TMP_TEST)
    os.mkdir(TMP_TEST + "/images")
    os.mkdir(TMP_TEST + "/labels")

    shutil.rmtree(TMP_VAL, ignore_errors=False, onerror=None)
    os.mkdir(TMP_VAL)
    os.mkdir(TMP_VAL + "/images")
    os.mkdir(TMP_VAL + "/labels")

    pad(TRAIN_PATH + '/images', TMP_TRAIN + '/images', already_padded)
    pad(TRAIN_PATH + '/labels', TMP_TRAIN + '/labels', already_padded)

    pad(TEST_PATH + '/images', TMP_TEST + '/images',already_padded)
    pad(TEST_PATH + '/labels', TMP_TEST + '/labels',already_padded)

    pad(VAL_PATH + '/images', TMP_VAL + '/images',already_padded)
    pad(VAL_PATH + '/labels', TMP_VAL + '/labels',already_padded)


    data_gen_args = dict()
    train_generator = trainGenerator(train_batch, TMP_TRAIN, 'images', 'labels', data_gen_args, save_to_dir = None, target_size=(608,576))
    test_generator = testGenerator2(test_batch, TMP_TEST, 'images', 'labels', data_gen_args, save_to_dir = None, target_size=(608,576))

    if model_name == "vanilla":
        model = UNet(input_size=(608,576,1))
    elif model_name == "attention":
        model = AttentionUNet(input_size=(608,576,1))

    if initial_model_path != None:
        model.load_weights(initial_model_path)

    model_checkpoint = ModelCheckpoint(MODEL_PATH + "/" + save_name +".hdf5", monitor='loss',verbose=1, save_best_only=True)

    model_history = model.fit_generator(train_generator, steps_per_epoch=num_train//train_batch, \
                                        epochs=epoch, callbacks=[model_checkpoint],\
                                        validation_data=test_generator, validation_steps=num_test//test_batch)

    log_file = open(LOG_PATH + "/log_{}.pkl".format(save_name), "wb")#history file
    pickle.dump(model_history.history, log_file)
    log_file.close()

    test_generator_2 = testGenerator(TMP_TEST, target_size=(608,576))
    results = model.predict_generator(test_generator_2,verbose=1)

    shutil.rmtree(TMP_RESULT, ignore_errors=False, onerror=None)
    os.mkdir(TMP_RESULT)
    saveResult_drive(TMP_RESULT, results)

    os.mkdir(RESULT_PATH + '/' + save_name)
    crop(TMP_RESULT, RESULT_PATH + '/' + save_name)

    threshold(RESULT_PATH + '/' + save_name)

    if os.path.isdir(RESULT_PATH + "/download"):
        shutil.rmtree(RESULT_PATH + '/download', ignore_errors=False, onerror=None)

    os.mkdir(RESULT_PATH + "/download")
    #shutil.rmtree(RESULT_PATH + '/download', ignore_errors=False, onerror=None)
    #os.mkdir(RESULT_PATH + '/download')
    set_order(RESULT_PATH + '/' + save_name, RESULT_PATH + '/download')



def train_loop(save_name, num_train, num_test, initial_model_path,\
               train_batch = 3, test_batch = 3, epoch = 5, already_padded = False,\
               model_name = "vanilla"):


    shutil.rmtree(TMP_TRAIN, ignore_errors=False, onerror=None)
    os.mkdir(TMP_TRAIN)
    os.mkdir(TMP_TRAIN + "/images")
    os.mkdir(TMP_TRAIN + "/labels")

    shutil.rmtree(TMP_TEST, ignore_errors=False, onerror=None)
    os.mkdir(TMP_TEST)
    os.mkdir(TMP_TEST + "/images")
    os.mkdir(TMP_TEST + "/labels")

    shutil.rmtree(TMP_VAL, ignore_errors=False, onerror=None)
    os.mkdir(TMP_VAL)
    os.mkdir(TMP_VAL + "/images")
    os.mkdir(TMP_VAL + "/labels")

    pad(TRAIN_PATH + '/images', TMP_TRAIN + '/images', already_padded)
    pad(TRAIN_PATH + '/labels', TMP_TRAIN + '/labels', already_padded)

    pad(TEST_PATH + '/images', TMP_TEST + '/images',already_padded)
    pad(TEST_PATH + '/labels', TMP_TEST + '/labels',already_padded)

    pad(VAL_PATH + '/images', TMP_VAL + '/images',already_padded)
    pad(VAL_PATH + '/labels', TMP_VAL + '/labels',already_padded)


    data_gen_args = dict()
    train_generator = trainGenerator(train_batch, TMP_TRAIN, 'images', 'labels', data_gen_args, save_to_dir = None, target_size=(608,576))
    test_generator = testGenerator2(test_batch, TMP_TEST, 'images', 'labels', data_gen_args, save_to_dir = None, target_size=(608,576))

    for epoch in range(epochs):

        if model_name == "vanilla":
            model = UNet(input_size=(608,576,1))
        elif model_name == "attention":
            model = AttentionUNet(input_size=(608,576,1))

        if epoch != 0:
            model.load_weights(f'{MODEL_PATH}/{save_name +"_"+ str(epoch-1)}.hdf5')
        else:
            if initial_model_name != None:
                model.load_weights(initial_model_path)


        model_history = model.fit_generator(train_generator, steps_per_epoch=num_train//train_batch, \
                                            epochs=1, callbacks=[model_checkpoint],\
                                            validation_data=test_generator, validation_steps=num_test//test_batch)


        log_file = open(LOG_PATH + "/log_{}.pkl".format(save_name + str(epoch)), "wb")#history file
        pickle.dump(model_history.history, log_file)
        log_file.close()

        test_generator_2 = testGenerator(TMP_TEST, target_size=(608,576))
        results = model.predict_generator(test_generator_2,verbose=1)


        shutil.rmtree(TMP_RESULT, ignore_errors=False, onerror=None)
        os.mkdir(TMP_RESULT)
        saveResult_drive(TMP_RESULT, results)

        os.mkdir(RESULT_PATH + '/' + save_name + str(epoch))
        crop(TMP_RESULT, RESULT_PATH + '/' + save_name + str(epoch))

        threshold(RESULT_PATH + '/' + save_name + str(epoch))

        os.mkdir(RESULT_PATH + '/download' + "_"+save_name + str(epoch))
        set_order(RESULT_PATH + '/' + save_name + str(epoch), RESULT_PATH + '/download' + "_"+save_name + str(epoch))

        mean_dice_coef = mean_dice(TEST_PATH   + '/labels',
                                   RESULT_PATH + '/download_' + save_name + str(epoch))

        print(f"\nMean dice coeff at epoch {epoch}: {mean_dice_coef}\n\n")
        print("-"*20)




def train_kfold_stare(epoch, start, train_batch_size, \
                      test_batch_size, train_sample_number, \
                      test_sample_number, initial_model_path, \
                      k=5,show_samples=False, model_name = "vanilla"):

    assert 20 % k ==0, "Number of images divided by fold number must be integer."
    NOF_PLOTS = 0

    for i in range(start, int(20/k), 1):
        test_images_temp = [j for j in range(k)]
        test_images_temp_2 = [a + i*k for a in test_images_temp]
        test_images = [str(a) for a in test_images_temp_2] #our test ids
        print("Test images: {}".format(test_images))


        shutil.rmtree(KFOLD_TEMP_TRAIN, ignore_errors=False, onerror=None)
        os.mkdir(KFOLD_TEMP_TRAIN)
        os.mkdir(KFOLD_TEMP_TRAIN + "/images")
        os.mkdir(KFOLD_TEMP_TRAIN + "/labels")

        shutil.rmtree(KFOLD_TEMP_TEST, ignore_errors=False, onerror=None)
        os.mkdir(KFOLD_TEMP_TEST)
        os.mkdir(KFOLD_TEMP_TEST + "/images")
        os.mkdir(KFOLD_TEMP_TEST + "/labels")

        for test_image in test_images: #allocates test images into the path
            src = TRAIN_PATH_IMG + "/" + test_image + ".png"
            shutil.copy(src, KFOLD_TEMP_TEST + "/images")

            src = TRAIN_PATH_MASK + "/" + test_image + ".png"
            shutil.copy(src, KFOLD_TEMP_TEST + "/labels")

        for img in sorted(os.listdir(TRAIN_PATH_IMG)): #allocates train images into the path
            img_splitted_1 = img.split("_")
            img_splitted_2 = img.split(".")
            if (img_splitted_1[-1].split(".")[0] not in test_images) and (img_splitted_2[0] not in test_images):
                src = TRAIN_PATH_IMG + "/" + img
                shutil.copy(src, KFOLD_TEMP_TRAIN + "/images")

                src = TRAIN_PATH_MASK + "/" + img
                shutil.copy(src, KFOLD_TEMP_TRAIN + "/labels")


        data_gen_args = dict()
        train_generator = trainGenerator(train_batch_size,KFOLD_TEMP_TRAIN,\
                                         'images','labels',data_gen_args,\
                                         save_to_dir = None, target_size = (608,704))

        test_generator = testGenerator2(test_batch_size,KFOLD_TEMP_TEST,\
                                        'images','labels',data_gen_args,\
                                        save_to_dir = None, target_size = (608,704))


        if model_name == "vanilla":
            model = UNet(input_size=(608,704,1))
        elif model_name == "attention":
            model = AttentionUNet(input_size=(608,704,1))

        if initial_model_path != None:
            model.load_weights(initial_model_path)

        model_checkpoint = ModelCheckpoint(CKPTS_PATH + "/fold_{}_unet_stare.hdf5".format(i), monitor='loss',verbose=1, save_best_only=True)

        model_history = model.fit_generator(train_generator,steps_per_epoch=train_sample_number//train_batch_size,\
                                            epochs=epoch, callbacks=[model_checkpoint],validation_data=test_generator,\
                                            validation_steps=test_sample_number//test_batch_size)

        log_file = open(LOG_PATH_K + "/log_fold_{}.pkl".format(i), "wb")#history file
        pickle.dump(model_history.history, log_file)
        log_file.close()

        test_generator_2 = testGenerator(KFOLD_TEMP_TEST, target_size = (608,704))
        results = model.predict_generator(test_generator_2,k,verbose=1)
        saveResult(i,k,RESULTS_PATH,results)

        del model

        if show_samples:
            fig, axs = plt.subplots(k + NOF_PLOTS,3,figsize=(17,17))
            for idx,item in enumerate(sorted(os.listdir(RESULTS_PATH))):
                item_without_predict = item.split("_")[0] + ".png"
                img_real = cv2.imread(TRAIN_PATH_IMG + "/" + item_without_predict)
                img_real = cv2.cvtColor(img_real, cv2.COLOR_BGR2RGB)
                #print(TRAIN_PATH_IMG + "/" + item_without_predict)
                #print(os.path.isfile(TRAIN_PATH_IMG + "/" + item_without_predict))
                img_ground_truth = cv2.imread(TRAIN_PATH_MASK + "/" + item_without_predict)
                img_ground_truth = cv2.cvtColor(img_ground_truth, cv2.COLOR_BGR2RGB)
                img_predict = cv2.imread(RESULTS_PATH + "/" + item)
                img_predict = cv2.cvtColor(img_predict, cv2.COLOR_BGR2RGB)

                axs[idx,0].imshow(img_real)
                axs[idx,0].title.set_text('image_{}'.format(item_without_predict))
                axs[idx,1].imshow(img_ground_truth)
                axs[idx,1].title.set_text('ground truth')
                axs[idx,2].imshow(img_predict)
                axs[idx,2].title.set_text('predicted')
            plt.show()
            NOF_PLOTS += k

In [ ]:
!pip install opencv-python

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from vanilla_unet import UNet  # GitHub theke upload kora file

In [ ]:
# Input folder (apnar images)
input_folder = '/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset'  

# Output folder (segmentation masks save hobe)
output_folder = '/kaggle/working/segmented_masks/'  
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Load UNet model
model = UNet(pretrained_weights=None, input_size=(608,576,1))  

# Optional: jodi pre-trained weights thake
# model.load_weights('/kaggle/input/your_weights/unet_weights.h5')

In [ ]:
for img_name in tqdm(os.listdir(input_folder)):
    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
        img_path = os.path.join(input_folder, img_name)
        
        # 1️⃣ Read image in grayscale
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # 2️⃣ Resize to model input
        img_resized = cv2.resize(img, (576,608))
        
        # 3️⃣ Normalize
        img_norm = img_resized / 255.0
        
        # 4️⃣ Add batch and channel dimension
        img_input = np.expand_dims(img_norm, axis=(0,-1))  # shape: (1,608,576,1)
        
        # 5️⃣ Predict mask
        pred_mask = model.predict(img_input)[0,:,:,0]
        
        # 6️⃣ Threshold → binary mask
        pred_mask = (pred_mask > 0.5).astype(np.uint8) * 255
        
        # 7️⃣ Save mask
        save_path = os.path.join(output_folder, img_name)
        cv2.imwrite(save_path, pred_mask)

print("Segmentation complete. Masks saved at:", output_folder)